# Aperture — AI vs Real Detector Training

Train an EfficientNet-B0 detector on CIFAKE (60k images, 32x32 upsampled to 224x224).

**Targets on the held-out test split:** accuracy ≥ 0.90, AUC ≥ 0.95, F1 ≥ 0.90.

Works on both Colab (with GPU) and a local machine with a CUDA GPU.
The cells below auto-detect Colab and set up Drive + Kaggle download accordingly.

In [ ]:
# --- environment setup (Colab-aware) ---
import os, sys, pathlib

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    APERTURE_ROOT = pathlib.Path('/content/drive/MyDrive/Aperture')
    APERTURE_ROOT.mkdir(parents=True, exist_ok=True)
    !pip install -q tqdm scikit-learn matplotlib seaborn kaggle
    if not (APERTURE_ROOT / 'Aperture' / '__init__.py').exists():
        raise FileNotFoundError(
            f'Aperture repo not found at {APERTURE_ROOT}. '
            'git clone or upload the repo there, then re-run this cell.')
    os.chdir(APERTURE_ROOT)
else:
    os.chdir(pathlib.Path().resolve().parent)

print('cwd:', os.getcwd())
sys.path.insert(0, os.getcwd())

In [ ]:
# --- download CIFAKE via Kaggle (one-time) ---
# Looks for kaggle.json in: ~/.kaggle/, Drive root (MyDrive/kaggle.json),
# Drive Aperture/ (MyDrive/Aperture/kaggle.json), or the repo root.
import pathlib, shutil
DATA_DIR = pathlib.Path('data/cifake')
if not (DATA_DIR / 'train' / 'REAL').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    home_token = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
    if not home_token.exists():
        candidates = [
            pathlib.Path('/content/drive/MyDrive/kaggle.json'),
            pathlib.Path('/content/drive/MyDrive/Aperture/kaggle.json'),
            pathlib.Path('kaggle.json'),
        ]
        token = next((p for p in candidates if p.exists()), None)
        if token is None:
            raise FileNotFoundError(
                'kaggle.json not found in Drive root or repo root.')
        home_token.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(token, home_token)
        home_token.chmod(0o600)
        print('Using kaggle token from:', token)
    !kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p data/ --unzip
    for split in ('train', 'test'):
        src = pathlib.Path('data') / split
        if src.exists():
            shutil.move(str(src), str(DATA_DIR / split))
    for split in ('train', 'test'):
        for cls in ('REAL', 'FAKE'):
            n = len(list((DATA_DIR / split / cls).iterdir()))
            print(f'{split:5s} {cls}: {n}')
else:
    print('CIFAKE already present at', DATA_DIR)

In [ ]:
# --- imports ---
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

from Aperture.ai_detector.train import TrainConfig, train
from Aperture.ai_detector.infer import AIDetector
from Aperture.ai_detector.dataset import CIFakeDataset

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# --- run training ---
config = TrainConfig(
    data_root='data/cifake',
    model_name='efficientnet_b0',
    epochs=10,
    batch_size=32,
    lr=1e-4,
    weight_decay=0.01,
    num_workers=2,
)
best_path, history = train(config)
print('Best checkpoint:', best_path)

In [ ]:
# --- inline training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(epochs, history['val_acc'], label='acc')
axes[1].plot(epochs, history['val_f1'], label='f1')
axes[1].plot(epochs, history['val_auc'], label='auc')
axes[1].set_title('Validation metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()
fig.tight_layout()
plt.show()
print('Final val metrics:',
      {k: round(history[k][-1], 4) for k in ('val_loss','val_acc','val_f1','val_auc')})

In [ ]:
# --- sample predictions ---
detector = AIDetector(best_path)
val_ds = CIFakeDataset('data/cifake/test', transform=None)
random.seed(0)
samples = random.sample(val_ds.samples, 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (path, label) in zip(axes.flat, samples):
    img = Image.open(path).convert('RGB')
    pred = detector.predict(img)
    truth = 'REAL' if label == 0 else 'FAKE'
    correct = (pred['label'].upper() == truth)
    color = 'green' if correct else 'red'
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f"truth={truth} | pred={pred['label'].upper()} ({pred['confidence']:.2f})",
        color=color, fontsize=10)
fig.tight_layout()
plt.show()